# EDA — Part A: Basic Inspection

**Goal:** Know the data before modelling it.  
This notebook characterises the raw FER-2013 CSV: shape, dtypes, missing values,
pixel value ranges, label distribution, and Usage split counts.

Every anomaly found here feeds directly into the cleaning and preprocessing stages
(`config.yaml → cleaning.*` and `preprocessing.*`).

**Run order:** execute cells top-to-bottom.  
The dataset must be present in `data/` — run `scripts/train.py` with `stages.train: false`
to trigger the download stage only, or place `train.csv` manually.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable from the notebooks/ subdirectory
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher

cfg = load_config(ROOT / "config.yaml")
setup_logging(cfg)

DATA_DIR  = ROOT / cfg["paths"]["data_dir"]
CSV_PATH  = DATA_DIR / cfg["data"]["primary_csv"]

print(f"Data directory : {DATA_DIR}")
print(f"Primary CSV    : {CSV_PATH}")
print(f"CSV exists     : {CSV_PATH.exists()}")

## 1. Load the raw CSV

We load with `pandas.read_csv` here (not `Fer2013Fetcher`) because the EDA targets the
**raw string representation** — we want to inspect the `pixels` column as text before parsing.
`Fer2013Fetcher` is used in §6 to verify the parsed array shapes.

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
df.head()

## 2. Schema — dtypes and column overview

In [ ]:
df.info()

In [ ]:
print("Column dtypes:")
print(df.dtypes)
print()
print("Expected:")
print("  emotion  → int64  (label 0–6)")
print("  pixels   → object (space-separated pixel string)")
print("  Usage    → object (Training / PublicTest / PrivateTest)")

## 3. Summary statistics

In [ ]:
df.describe(include="all")

## 4. Missing value audit

Any NaN, empty string, or whitespace-only cell in `pixels` would cause `np.fromstring` to fail
silently or raise — catch them here before parsing.

In [ ]:
nan_counts = df.isna().sum()
print("NaN counts per column:")
print(nan_counts)
print()
if nan_counts.sum() == 0:
    print("✓ No NaN values found.")
else:
    print(f"⚠ {nan_counts.sum()} NaN values found — see cleaning stage.")

In [ ]:
# Whitespace-only or empty strings in pixels (not caught by isna)
empty_pixels = df["pixels"].str.strip().eq("").sum()
print(f"Empty / whitespace-only pixels rows: {empty_pixels}")
if empty_pixels == 0:
    print("✓ No empty pixel strings.")

## 5. Pixel-count sanity check

Every row must encode exactly **2 304 values** (48 × 48).  
A mismatch means a corrupt or truncated row that would silently produce a wrong-shaped array.

In [ ]:
pixel_counts = df["pixels"].str.split().str.len()

print("Pixel-count distribution:")
print(pixel_counts.value_counts().sort_index())
print()

bad_rows = df[pixel_counts != 2304]
print(f"Rows with pixel count ≠ 2304: {len(bad_rows)}")
if len(bad_rows) == 0:
    print("✓ All rows have exactly 2304 pixel values.")
else:
    print("⚠ Malformed rows found:")
    print(bad_rows[["emotion", "Usage"]].assign(pixel_count=pixel_counts[bad_rows.index]))

## 6. Pixel intensity statistics

Use `Fer2013Fetcher` to get parsed arrays, then compute intensity stats across
the full training split.  Pixel values should lie in **[0, 255]** as uint8.

In [ ]:
fetcher = Fer2013Fetcher(cfg)

images_train, labels_train = fetcher.fetch("Training")
images_val,   labels_val   = fetcher.fetch("PublicTest")
images_test,  labels_test  = fetcher.fetch("PrivateTest")

print(f"Training   — images: {images_train.shape}, labels: {labels_train.shape}")
print(f"Validation — images: {images_val.shape},   labels: {labels_val.shape}")
print(f"Test       — images: {images_test.shape},  labels: {labels_test.shape}")

In [ ]:
all_pixels = images_train.astype(np.float32)

print("Training pixel intensity statistics:")
print(f"  dtype   : {images_train.dtype}")
print(f"  min     : {all_pixels.min():.1f}")
print(f"  max     : {all_pixels.max():.1f}")
print(f"  mean    : {all_pixels.mean():.2f}")
print(f"  std     : {all_pixels.std():.2f}")
print(f"  median  : {np.median(all_pixels):.1f}")
print()
if all_pixels.min() >= 0 and all_pixels.max() <= 255:
    print("✓ All pixel values in valid range [0, 255].")
else:
    print("⚠ Out-of-range pixel values detected.")

## 7. Label distribution

In [ ]:
EMOTION_LABELS = {
    0: "Angry",
    1: "Disgust",
    2: "Fear",
    3: "Happy",
    4: "Sad",
    5: "Surprise",
    6: "Neutral",
}

label_counts = df["emotion"].value_counts().sort_index()
label_pct    = label_counts / len(df) * 100

print("Label distribution (full dataset):")
print(f"{'Code':<6} {'Label':<12} {'Count':>7} {'%':>7}")
print("-" * 36)
for code, count in label_counts.items():
    print(f"{code:<6} {EMOTION_LABELS[code]:<12} {count:>7,} {label_pct[code]:>6.1f}%")
print("-" * 36)
print(f"{'Total':<18} {len(df):>7,} {'100.0%':>7}")
print()

unique_labels = df["emotion"].unique()
print(f"Unique label values : {sorted(unique_labels)}")
out_of_range = df[~df["emotion"].between(0, 6)]
print(f"Out-of-range labels : {len(out_of_range)}")
if len(out_of_range) == 0:
    print("✓ All labels in valid range [0, 6].")

## 8. Usage split counts

In [ ]:
usage_counts = df["Usage"].value_counts()
usage_pct    = usage_counts / len(df) * 100

print("Usage split distribution:")
print(f"{'Split':<16} {'Count':>7} {'%':>7}")
print("-" * 33)
for split, count in usage_counts.items():
    print(f"{split:<16} {count:>7,} {usage_pct[split]:>6.1f}%")
print("-" * 33)
print(f"{'Total':<16} {len(df):>7,} {'100.0%':>7}")
print()

unexpected_splits = set(df["Usage"].unique()) - {"Training", "PublicTest", "PrivateTest"}
if not unexpected_splits:
    print("✓ Only expected Usage values present.")
else:
    print(f"⚠ Unexpected Usage values: {unexpected_splits}")

## 9. Per-split label distribution

Checks whether class imbalance is consistent across splits (it should be,
since FER-2013 was split randomly, not stratified).

In [ ]:
split_label = (
    df.groupby(["Usage", "emotion"])
    .size()
    .unstack(fill_value=0)
    .rename(columns=EMOTION_LABELS)
)
split_label

## 10. Findings summary

| Check | Result | Action |
|---|---|---|
| NaN values | — | — |
| Empty pixel strings | — | — |
| Pixel count ≠ 2304 | — | — |
| Pixel range [0, 255] | — | — |
| Label range [0, 6] | — | — |
| Unexpected Usage values | — | — |
| Class imbalance | Disgust ≈ 1.5 % vs Happy ≈ 25 % | `cleaning.imbalance_strategy: class_weight` |

> Fill in the **Result** column after running the notebook on the actual dataset.  
> Each anomaly that requires a fix must become a named, switchable option in `config.yaml`
> (see CONTRIBUTING.md §3 — Ablation-Driven Architecture).